# 05 — Stochastic / Subsampled Cubic Regularization

Compare full-batch CR, subsampled CR (varying batch sizes), SGD, and Adam on synthetic logistic regression. Primary metric: accuracy vs **wall-clock time**.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from cubic_reg.problems import make_synthetic_logistic
from cubic_reg.solvers import cr, stochastic

%matplotlib inline

In [ ]:
prob = make_synthetic_logistic(n_samples=1200, n_features=60, seed=0)
x0 = np.zeros(prob.dim)

r_full = cr.minimize(prob, x0=x0, M=1.0, eps=1e-4, max_iter=40)
r_sc_small = stochastic.minimize(prob, x0=x0, M=1.0, eps=1e-3, max_iter=60, batch_grad=64, batch_hess=16)
r_sc_med = stochastic.minimize(prob, x0=x0, M=1.0, eps=1e-3, max_iter=60, batch_grad=256, batch_hess=64)
r_sgd = stochastic.minimize_sgd(prob, x0=x0, max_iter=300, batch=64, lr=0.15, eps=1e-3)
r_adam = stochastic.minimize_adam(prob, x0=x0, max_iter=300, batch=64, lr=0.05, eps=1e-3)

methods = {
    "Full CR": r_full,
    "SubCR bg=64": r_sc_small,
    "SubCR bg=256": r_sc_med,
    "SGD": r_sgd,
    "Adam": r_adam,
}
for name, r in methods.items():
    print(f"{name:14} time={r.time_sec:.3f}s nit={r.nit:4d} ||g||={r.grad_norm:.3e} f={r.f:.5f}")

In [ ]:
plt.figure(figsize=(8, 4))
for name, r in methods.items():
    # approximate time axis by uniform spacing of total wall time
    t = np.linspace(0, r.time_sec, num=len(r.history_grad_norm))
    plt.semilogy(t, np.maximum(r.history_grad_norm, 1e-16), label=name)
plt.xlabel("wall time (s)"); plt.ylabel(r"$\|\nabla f\|$"); plt.legend()
plt.title("Stochastic methods: grad norm vs time"); plt.grid(True, alpha=0.3); plt.show()